# BirdCLEF 2026 SED Baseline

Notebook com pipeline mais proximo da competicao real: janelas de 5 segundos, treinamento multilabel em `train_soundscapes_labels.csv`, `mel spectrogram` e modelo em PyTorch inspirado na logica de SED com `EfficientNet-B0`.

## 1. Importacoes e configuracao

In [ ]:
from __future__ import annotations

import os
import random
from collections import defaultdict
from dataclasses import dataclass
from pathlib import Path

import kagglehub
import librosa
import numpy as np
import pandas as pd
import timm
import torch
import torch.nn as nn
import torchaudio
from kagglehub.exceptions import UnauthenticatedError
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

RANDOM_SEED = 42
SAMPLE_RATE = 32000
N_MELS = 128
N_FFT = 2048
HOP_LENGTH = 512
F_MIN = 20
F_MAX = 16000

WINDOW_SECONDS = 5.0
CONTEXT_SECONDS = 20.0
BATCH_SIZE = 8
EPOCHS = 3
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
VAL_SIZE = 0.2
MAX_TRAIN_WINDOWS = 2000
MAX_VAL_WINDOWS = 600
USE_PRETRAINED_BACKBONE = False
APPLY_TEMPORAL_SMOOTHING = True

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

print(f'Device: {DEVICE}')

## 2. Download do dataset

In [ ]:
try:
    DATASET_DIR = Path(kagglehub.competition_download('birdclef-2026')).resolve()
    print('Path to competition files:', DATASET_DIR)
except UnauthenticatedError:
    raise RuntimeError(
        'Kaggle authentication is required before downloading BirdCLEF 2026. '
        'Run `kagglehub login` or configure your Kaggle credentials first.'
    )

## 3. Funcoes auxiliares

In [ ]:
@dataclass
class WindowTarget:
    filename: str
    end_time: int
    target: np.ndarray


def resolve_dataset_dir(dataset_dir: str | Path | None = None) -> Path:
    candidates: list[Path] = []

    if dataset_dir is not None:
        candidates.append(Path(dataset_dir).expanduser())

    env_dir = os.getenv('BIRDCLEF_2026_DIR')
    if env_dir:
        candidates.append(Path(env_dir).expanduser())

    if 'DATASET_DIR' in globals():
        candidates.append(Path(DATASET_DIR))

    candidates.extend([
        Path.cwd() / 'birdclef-2026',
        Path.cwd() / 'data' / 'birdclef-2026',
    ])

    for candidate in candidates:
        resolved = candidate.resolve()
        if resolved.exists():
            return resolved

    searched = '\n'.join(f'- {path}' for path in candidates) or '- <none>'
    raise FileNotFoundError(
        'BirdCLEF 2026 dataset directory not found. Checked:\n'
        f'{searched}'
    )


def summarize_dataset(dataset_dir: Path) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame | None, pd.DataFrame | None]:
    train_csv = pd.read_csv(dataset_dir / 'train.csv')
    taxonomy_csv = pd.read_csv(dataset_dir / 'taxonomy.csv')

    soundscape_labels_path = dataset_dir / 'train_soundscapes_labels.csv'
    soundscape_labels = pd.read_csv(soundscape_labels_path) if soundscape_labels_path.exists() else None

    sample_submission_path = dataset_dir / 'sample_submission.csv'
    sample_submission = pd.read_csv(sample_submission_path) if sample_submission_path.exists() else None

    print(f'Dataset directory: {dataset_dir}')
    print(f'train.csv rows: {len(train_csv):,}')
    print(f'taxonomy.csv rows: {len(taxonomy_csv):,}')
    print(f"Unique classes in train.csv: {train_csv['primary_label'].nunique():,}")

    if soundscape_labels is not None:
        print(f'train_soundscapes_labels.csv rows: {len(soundscape_labels):,}')
    else:
        print('train_soundscapes_labels.csv not found.')

    if sample_submission is not None:
        print(f'sample_submission.csv rows: {len(sample_submission):,}')
    else:
        print('sample_submission.csv not found.')

    return train_csv, taxonomy_csv, soundscape_labels, sample_submission


def build_species_list(train_df: pd.DataFrame, sample_submission: pd.DataFrame | None) -> list[str]:
    if sample_submission is not None and len(sample_submission.columns) > 1:
        return sorted(sample_submission.columns[1:].tolist())
    return sorted(train_df['primary_label'].dropna().unique().tolist())


def resolve_audio_path(audio_dir: Path, filename: str) -> Path:
    candidate = audio_dir / filename
    if candidate.exists():
        return candidate

    if not Path(filename).suffix:
        with_ogg = audio_dir / f'{filename}.ogg'
        if with_ogg.exists():
            return with_ogg

    raise FileNotFoundError(f'Audio file not found for {filename}')


def parse_row_id(row_id: str) -> tuple[str, int]:
    filename, end_time = row_id.rsplit('_', 1)
    return filename, int(end_time)


def normalize_soundscape_labels(labels_df: pd.DataFrame | None, species_list: list[str]) -> pd.DataFrame:
    if labels_df is None or labels_df.empty:
        raise RuntimeError('train_soundscapes_labels.csv is required for the multilabel pipeline.')

    df = labels_df.copy()
    species_to_idx = {species: idx for idx, species in enumerate(species_list)}
    wide_species_columns = [col for col in df.columns if col in species_to_idx]

    if {'filename', 'end_time', 'primary_label'}.issubset(df.columns):
        aggregated: dict[tuple[str, int], np.ndarray] = defaultdict(
            lambda: np.zeros(len(species_list), dtype=np.float32)
        )
        for row in df[['filename', 'end_time', 'primary_label']].dropna().itertuples(index=False):
            key = (str(row.filename), int(row.end_time))
            if row.primary_label in species_to_idx:
                aggregated[key][species_to_idx[row.primary_label]] = 1.0

        rows = []
        for (filename, end_time), target in aggregated.items():
            row = {'filename': filename, 'end_time': end_time}
            row.update({species: float(target[idx]) for species, idx in species_to_idx.items()})
            rows.append(row)
        normalized = pd.DataFrame(rows)
    elif {'filename', 'end_time'}.issubset(df.columns) and wide_species_columns:
        normalized = df[['filename', 'end_time', *wide_species_columns]].copy()
    elif 'row_id' in df.columns and wide_species_columns:
        filenames = []
        end_times = []
        for row_id in df['row_id'].astype(str):
            filename, end_time = parse_row_id(row_id)
            filenames.append(filename)
            end_times.append(end_time)
        normalized = df[['row_id', *wide_species_columns]].copy()
        normalized['filename'] = filenames
        normalized['end_time'] = end_times
        normalized = normalized.drop(columns=['row_id'])
    else:
        raise ValueError(
            'Unsupported schema for train_soundscapes_labels.csv. '
            'Expected long format with filename/end_time/primary_label or wide format with species columns.'
        )

    for species in species_list:
        if species not in normalized.columns:
            normalized[species] = 0.0

    normalized['filename'] = normalized['filename'].astype(str)
    normalized['end_time'] = normalized['end_time'].astype(int)
    normalized = normalized[['filename', 'end_time', *species_list]].fillna(0.0)
    normalized[species_list] = normalized[species_list].astype(np.float32)
    normalized = normalized.sort_values(['filename', 'end_time']).reset_index(drop=True)
    return normalized


def build_window_targets(
    labels_df: pd.DataFrame,
    species_list: list[str],
    audio_dir: Path,
) -> list[WindowTarget]:
    targets: list[WindowTarget] = []
    for row in labels_df.itertuples(index=False):
        try:
            resolve_audio_path(audio_dir, row.filename)
        except FileNotFoundError:
            continue

        target = np.asarray([getattr(row, species) for species in species_list], dtype=np.float32)
        if target.sum() == 0:
            continue

        targets.append(WindowTarget(filename=row.filename, end_time=int(row.end_time), target=target))

    if not targets:
        raise RuntimeError('No valid soundscape windows were found for training.')

    return targets


def split_targets_by_filename(
    targets: list[WindowTarget],
    val_size: float,
) -> tuple[list[WindowTarget], list[WindowTarget]]:
    grouped: dict[str, list[WindowTarget]] = defaultdict(list)
    for target in targets:
        grouped[target.filename].append(target)

    filenames = sorted(grouped)
    train_names, val_names = train_test_split(filenames, test_size=val_size, random_state=RANDOM_SEED)
    train_targets = [target for name in train_names for target in grouped[name]]
    val_targets = [target for name in val_names for target in grouped[name]]
    return train_targets, val_targets


def limit_targets(targets: list[WindowTarget], max_items: int | None) -> list[WindowTarget]:
    if max_items is None or len(targets) <= max_items:
        return targets
    return targets[:max_items]


class BirdCLEFWindowDataset(Dataset):
    def __init__(self, targets: list[WindowTarget], audio_dir: Path, species_list: list[str]):
        self.targets = targets
        self.audio_dir = audio_dir
        self.species_list = species_list
        self.window_samples = int(WINDOW_SECONDS * SAMPLE_RATE)
        self.mel_transform = torchaudio.transforms.MelSpectrogram(
            sample_rate=SAMPLE_RATE,
            n_fft=N_FFT,
            hop_length=HOP_LENGTH,
            n_mels=N_MELS,
            f_min=F_MIN,
            f_max=F_MAX,
            power=2.0,
        )
        self.amplitude_to_db = torchaudio.transforms.AmplitudeToDB(stype='power', top_db=80)

    def __len__(self) -> int:
        return len(self.targets)

    def _load_window(self, target: WindowTarget) -> np.ndarray:
        audio_path = resolve_audio_path(self.audio_dir, target.filename)
        waveform, _ = librosa.load(audio_path, sr=SAMPLE_RATE, mono=True)
        end_sample = int(target.end_time * SAMPLE_RATE)
        start_sample = max(0, end_sample - self.window_samples)
        window = waveform[start_sample:end_sample]
        if len(window) < self.window_samples:
            window = np.pad(window, (self.window_samples - len(window), 0), mode='constant')
        return window.astype(np.float32)

    def __getitem__(self, index: int) -> tuple[torch.Tensor, torch.Tensor]:
        target = self.targets[index]
        window = self._load_window(target)
        waveform = torch.from_numpy(window).float().unsqueeze(0)
        mel = self.amplitude_to_db(self.mel_transform(waveform)).squeeze(0)
        label = torch.from_numpy(target.target)
        return mel, label


class AttentionPooling(nn.Module):
    def __init__(self, in_features: int):
        super().__init__()
        self.attention = nn.Sequential(
            nn.Linear(in_features, in_features),
            nn.Tanh(),
            nn.Linear(in_features, 1),
        )

    def forward(self, x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        weights = torch.softmax(self.attention(x), dim=1)
        return (x * weights).sum(dim=1), weights


class SEDModel(nn.Module):
    def __init__(
        self,
        num_classes: int,
        backbone_name: str = 'efficientnet_b0',
        pretrained: bool = False,
        in_chans: int = 1,
        drop_rate: float = 0.3,
        drop_path_rate: float = 0.1,
    ):
        super().__init__()
        self.backbone = timm.create_model(
            backbone_name,
            pretrained=pretrained,
            in_chans=in_chans,
            features_only=True,
            drop_rate=drop_rate,
            drop_path_rate=drop_path_rate,
        )

        with torch.no_grad():
            sample = torch.randn(1, in_chans, N_MELS, 320)
            last_feat = self.backbone(sample)[-1]
            self._feat_channels = last_feat.shape[1]
            self._feat_freq = last_feat.shape[2]

        fc_in = self._feat_channels * self._feat_freq
        self.framewise_head = nn.Sequential(
            nn.Dropout(drop_rate),
            nn.Linear(fc_in, num_classes),
        )
        self.attention_pool = AttentionPooling(fc_in)
        self.clipwise_head = nn.Sequential(
            nn.Dropout(drop_rate),
            nn.Linear(fc_in, num_classes),
        )

    def forward(self, x: torch.Tensor) -> dict[str, torch.Tensor]:
        if x.dim() == 3:
            x = x.unsqueeze(1)

        feat = self.backbone(x)[-1]
        batch_size, channels, freq_bins, frames = feat.shape
        feat = feat.reshape(batch_size, channels * freq_bins, frames).permute(0, 2, 1)

        framewise_logits = self.framewise_head(feat)
        pooled, attention_weights = self.attention_pool(feat)
        clipwise_logits = self.clipwise_head(pooled)

        return {
            'clipwise': clipwise_logits,
            'framewise': framewise_logits,
            'attention_weights': attention_weights,
        }


def multilabel_f1_from_logits(logits: torch.Tensor, targets: torch.Tensor, threshold: float = 0.5) -> float:
    probs = torch.sigmoid(logits).detach().cpu().numpy()
    truth = targets.detach().cpu().numpy()
    preds = (probs >= threshold).astype(np.int32)
    return float(f1_score(truth, preds, average='samples', zero_division=0))


def run_epoch(
    model: nn.Module,
    dataloader: DataLoader,
    criterion: nn.Module,
    optimizer: torch.optim.Optimizer | None = None,
) -> tuple[float, float]:
    is_train = optimizer is not None
    model.train(mode=is_train)

    total_loss = 0.0
    total_f1 = 0.0
    batches = 0

    for mels, targets in tqdm(dataloader, leave=False):
        mels = mels.to(DEVICE)
        targets = targets.to(DEVICE)

        with torch.set_grad_enabled(is_train):
            outputs = model(mels)
            clipwise_logits = outputs['clipwise']
            loss = criterion(clipwise_logits, targets)

            if is_train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

        total_loss += float(loss.item())
        total_f1 += multilabel_f1_from_logits(clipwise_logits, targets)
        batches += 1

    return total_loss / max(batches, 1), total_f1 / max(batches, 1)


def train_model(
    train_loader: DataLoader,
    val_loader: DataLoader,
    num_classes: int,
    epochs: int = EPOCHS,
) -> tuple[nn.Module, pd.DataFrame]:
    model = SEDModel(
        num_classes=num_classes,
        pretrained=USE_PRETRAINED_BACKBONE,
    ).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    criterion = nn.BCEWithLogitsLoss()

    history_rows = []
    best_state = None
    best_val_f1 = -1.0

    for epoch in range(1, epochs + 1):
        train_loss, train_f1 = run_epoch(model, train_loader, criterion, optimizer)
        val_loss, val_f1 = run_epoch(model, val_loader, criterion)

        history_rows.append({
            'epoch': epoch,
            'train_loss': train_loss,
            'train_f1_samples': train_f1,
            'val_loss': val_loss,
            'val_f1_samples': val_f1,
        })

        print(
            f'Epoch {epoch:02d} | '
            f'train_loss={train_loss:.4f} train_f1={train_f1:.4f} | '
            f'val_loss={val_loss:.4f} val_f1={val_f1:.4f}'
        )

        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}

    if best_state is not None:
        model.load_state_dict(best_state)

    history = pd.DataFrame(history_rows)
    return model, history


def compute_mel_tensor(waveform_np: np.ndarray) -> torch.Tensor:
    mel_transform = torchaudio.transforms.MelSpectrogram(
        sample_rate=SAMPLE_RATE,
        n_fft=N_FFT,
        hop_length=HOP_LENGTH,
        n_mels=N_MELS,
        f_min=F_MIN,
        f_max=F_MAX,
        power=2.0,
    ).to(DEVICE)
    amplitude_to_db = torchaudio.transforms.AmplitudeToDB(stype='power', top_db=80).to(DEVICE)

    waveform = torch.from_numpy(waveform_np).float().unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        mel = amplitude_to_db(mel_transform(waveform))
    return mel.unsqueeze(0)


def make_submission(
    model: nn.Module,
    soundscape_dir: Path,
    species_list: list[str],
    sample_submission: pd.DataFrame,
) -> pd.DataFrame:
    context_samples = int(CONTEXT_SECONDS * SAMPLE_RATE)
    predict_samples = int(WINDOW_SECONDS * SAMPLE_RATE)
    segments_per_context = int(CONTEXT_SECONDS / WINDOW_SECONDS)

    rows = []
    soundscape_files = sorted(soundscape_dir.glob('*.ogg'))

    model.eval()
    for soundscape_path in tqdm(soundscape_files, desc='Inference'):
        waveform, _ = librosa.load(soundscape_path, sr=SAMPLE_RATE, mono=True)
        total_samples = len(waveform)

        for ctx_start in range(0, total_samples, context_samples):
            chunk = waveform[ctx_start:ctx_start + context_samples]
            if len(chunk) < context_samples:
                chunk = np.pad(chunk, (0, context_samples - len(chunk)), mode='constant')

            mel_tensor = compute_mel_tensor(chunk)
            with torch.no_grad():
                framewise_logits = model(mel_tensor)['framewise'][0]

            segment_logits = torch.tensor_split(framewise_logits, segments_per_context, dim=0)
            for seg_idx, seg_logit in enumerate(segment_logits):
                if seg_logit.numel() == 0:
                    continue

                seg_probs = torch.sigmoid(seg_logit).mean(dim=0).cpu().numpy()
                seg_start = ctx_start + seg_idx * predict_samples
                if seg_start >= total_samples:
                    break

                end_time = (ctx_start + (seg_idx + 1) * predict_samples) // SAMPLE_RATE
                row = {'row_id': f'{soundscape_path.stem}_{end_time}'}
                row.update({species: float(seg_probs[idx]) for idx, species in enumerate(species_list)})
                rows.append(row)

    predictions = pd.DataFrame(rows)
    if predictions.empty:
        predictions = sample_submission.copy()
        predictions.iloc[:, 1:] = 0.0
        return predictions

    if APPLY_TEMPORAL_SMOOTHING and len(predictions) > 2:
        grouped_rows = defaultdict(list)
        for row in predictions.to_dict(orient='records'):
            grouped_rows[row['row_id'].rsplit('_', 1)[0]].append(row)

        smoothed_rows = []
        for soundscape_id, items in grouped_rows.items():
            for idx, item in enumerate(items):
                smoothed = {'row_id': item['row_id']}
                for species in species_list:
                    center = item[species]
                    left = items[idx - 1][species] if idx > 0 else center
                    right = items[idx + 1][species] if idx + 1 < len(items) else center
                    smoothed[species] = float(0.25 * left + 0.50 * center + 0.25 * right)
                smoothed_rows.append(smoothed)
        predictions = pd.DataFrame(smoothed_rows)

    expected_columns = sample_submission.columns.tolist()
    for species in expected_columns[1:]:
        if species not in predictions.columns:
            predictions[species] = 0.0

    predictions = predictions[expected_columns].copy()
    return predictions


## 4. Auditoria inicial do dataset

In [ ]:
dataset_dir = resolve_dataset_dir()
train_df, taxonomy_df, soundscape_labels_df, sample_submission_df = summarize_dataset(dataset_dir)

species_list = build_species_list(train_df, sample_submission_df)
train_soundscapes_dir = dataset_dir / 'train_soundscapes'
test_soundscapes_dir = dataset_dir / 'test_soundscapes'

normalized_labels_df = normalize_soundscape_labels(soundscape_labels_df, species_list)
window_targets = build_window_targets(normalized_labels_df, species_list, train_soundscapes_dir)

print(f'Scored species: {len(species_list):,}')
print(f'Train soundscape windows: {len(window_targets):,}')
print('train_soundscapes exists:', train_soundscapes_dir.exists())
print('test_soundscapes exists:', test_soundscapes_dir.exists())
normalized_labels_df.head()

## 5. Treino multilabel com mel spectrogram

In [ ]:
train_targets, val_targets = split_targets_by_filename(window_targets, VAL_SIZE)
train_targets = limit_targets(train_targets, MAX_TRAIN_WINDOWS)
val_targets = limit_targets(val_targets, MAX_VAL_WINDOWS)

print(f'Train windows: {len(train_targets):,}')
print(f'Validation windows: {len(val_targets):,}')

train_dataset = BirdCLEFWindowDataset(train_targets, train_soundscapes_dir, species_list)
val_dataset = BirdCLEFWindowDataset(val_targets, train_soundscapes_dir, species_list)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

model, history_df = train_model(train_loader, val_loader, num_classes=len(species_list), epochs=EPOCHS)
history_df

## 6. Geracao de submissao

In [ ]:
if sample_submission_df is None:
    raise RuntimeError('sample_submission.csv is required to build the final submission file.')

inference_dir = test_soundscapes_dir if test_soundscapes_dir.exists() else train_soundscapes_dir
submission_df = make_submission(model, inference_dir, species_list, sample_submission_df)

print(f'Submission shape: {submission_df.shape}')
print(submission_df.head())

## 7. Proximos ajustes

- A base agora usa a mesma ideia central do notebook do Kaggle: `mel spectrogram`, `EfficientNet-B0`, saida multilabel e inferencia por janelas temporais.
- O treino ainda esta simplificado para caber em um notebook local: `BCEWithLogitsLoss`, `clipwise` supervisionado e numero reduzido de janelas.
- Se voces tiverem GPU e mais tempo de treino, os proximos ganhos naturais sao aumentar `EPOCHS`, ampliar `MAX_TRAIN_WINDOWS`, salvar checkpoints e adicionar augmentations.